# Jouvence / TxGNN KG — état courant au 2026-06-24

Notebook read-only pour explorer l'état **actuel** du KG, des décisions de modélisation, de LaminDB, PyG, ReMap, mutations, clinical trials et des gaps.

Sources de vérité utilisées par défaut:

- Canonical KG: `/Users/jkobject/mnt/gcs/jouvencekb-kg/v2` / `gs://jouvencekb/kg/v2`
- Coverage docs: `docs/relation_coverage_current.md`, `docs/relation_backlog_prioritized.md`
- Phase mirrors: `todo.d/*.md`
- Board Kanban: `txgnn`

⚠️ Ce notebook ne promeut rien en canonical et ne modifie pas le KG.

## TL;DR état vérifié

- **Canonical KG edges**: 40 relations présentes, ~100.08M canonical edge rows.
- **Evidence**: 18 relations ont evidence files, 22 canonical relations n'ont pas encore d'evidence file.
- **Mutation**:
  - `mutation_affects_transcript`: canonical promoted + review accepted.
  - `mutation_in_gene`: canonical written/promoted, review-required; 2,599,525 edge/evidence/proof rows, validation endpoint/proof passée.
  - `mutation_overlaps_enhancer`: feature/context only, pas canonical edge.
- **ReMap / `tf_binds_enhancer`**:
  - Bounded staged pilot accepted: 1,224,536 edges / 6,356,561 evidence rows.
  - Full CRM support sidecar canonical feature promoted: 48,768,788 summary rows + 1,179 TF global rows.
  - Full edge/evidence not materialized: feasibility gate ~24.45B candidate rows / ~4.85TiB lower-bound parquet; route C = support-only until reduction policy.
  - Motif: policy planned, not yet a populated motif-overlap field.
  - Antibody/cell/biotype: ReMap biotype metadata exists in sample; CRM itself lacks native peak FK/antibody fields, so evidence must preserve metadata coverage/missingness.
- **LaminDB**: active instance has `lnschema_txgnn`; bounded live sync, not full 100M-edge sync.
- **PyG**: real HeteroData/GNN smoke artifacts exist; not full-KG production GNN.
- **Dataset/paper**: disconnected from default training graph; kept as metadata/provenance only.
- **Clinical trials**: active card to add phase/endpoint/failure-aware evidence + LLM text embeddings.

In [1]:
from pathlib import Path
import json, os, glob, textwrap
KG_ROOT = Path('/Users/jkobject/mnt/gcs/jouvencekb-kg/v2')
print('KG root exists:', KG_ROOT.exists(), KG_ROOT)
print('Top dirs:', sorted([p.name for p in KG_ROOT.iterdir()])[:20] if KG_ROOT.exists() else 'missing')

KG root exists: True /Users/jkobject/mnt/gcs/jouvencekb-kg/v2
Top dirs: ['_backups', '_promotion_staging', '_removed_relations_20260618', 'archive', 'edges', 'evidence', 'features', 'metadata', 'ml', 'nodes', 'proof', 'raw', 'staged', 'staging']


## Relation coverage table from docs

In [2]:
from pathlib import Path
import re
coverage = Path('docs/relation_coverage_current.md').read_text()
for line in coverage.splitlines()[11:25]:
    print(line)


- Declared active relations in schema: `67`
- Canonical edge relations present in `v2/edges`: `40`
- Canonical edge relations with matching `v2/evidence` file: `18`
- Canonical edge relations without `v2/evidence` file: `22`
- Declared relations not present as canonical edge files: `27`
- Staged-only/deferred declared relations with staged edge files: `18`
- Source-audit-only/deferred declared relations: `2`
- Feature/context-not-edge declared relations: `2`
- Schema-only/missing declared relations: `5`
- Intentionally retired active relations: `0`
- Candidate/not-active relations in `CANDIDATE_RELATIONS`: `2`
- Total canonical edge rows across current edge files: `100,080,390`



## Canonical edge/evidence counts from Parquet metadata
Lightweight: uses pyarrow metadata when possible; falls back to file presence.

In [3]:
from pathlib import Path
try:
    import pyarrow.parquet as pq
except Exception as e:
    pq = None
    print('pyarrow unavailable:', e)

def parquet_rows(path):
    if pq is None: return None
    path = Path(path)
    if path.is_file():
        return pq.ParquetFile(path).metadata.num_rows
    if path.is_dir():
        return sum(pq.ParquetFile(p).metadata.num_rows for p in path.glob('*.parquet'))
    return None

edges = sorted((KG_ROOT/'edges').glob('*.parquet')) if (KG_ROOT/'edges').exists() else []
evidence = sorted((KG_ROOT/'evidence').glob('*.parquet')) if (KG_ROOT/'evidence').exists() else []
print('edge files', len(edges), 'evidence files', len(evidence))
for p in edges[:10]:
    ev = KG_ROOT/'evidence'/p.name
    print(p.name, parquet_rows(p), 'evidence=', parquet_rows(ev) if ev.exists() else None)

edge files 40 evidence files 18


cell_line_derived_from_tissue.parquet 1092 evidence= None


cell_line_expresses_gene.parquet 20928056 evidence= None


cell_line_from_organism.parquet 1183 evidence= 1183


cell_type_expresses_gene.parquet 1561873 evidence= None


...

dataset_contains_tissue.parquet 27 evidence= None


disease_associated_gene.parquet 83339 evidence= 2928


disease_has_phenotype.parquet 241797 evidence= None


disease_involves_pathway.parquet 2296 evidence= 2296


disease_manifests_in_tissue.parquet 19 evidence= 29


## Mutation status

In [4]:
for rel in ['mutation_affects_transcript','mutation_in_gene','mutation_overlaps_enhancer']:
    e = KG_ROOT/'edges'/f'{rel}.parquet'
    ev = KG_ROOT/'evidence'/f'{rel}.parquet'
    proof = KG_ROOT/'proof'/f'{rel}_containment_proof.parquet'
    print(rel)
    print('  edge rows:', parquet_rows(e) if e.exists() else None, e.exists())
    print('  evidence rows:', parquet_rows(ev) if ev.exists() else None, ev.exists())
    if proof.exists(): print('  proof rows:', parquet_rows(proof))

mutation_affects_transcript
  edge rows: 2599922 True


  evidence rows: 2599922 True
mutation_in_gene
  edge rows: 2599525 True


  evidence rows: 2599525 True
  proof rows: 2599525
mutation_overlaps_enhancer


  edge rows: None False
  evidence rows: None False


## ReMap / tf_binds_enhancer

Important: `tf_binds_enhancer` est le bon nom de relation pour cette famille, mais le full graph n'est pas canonical-promoted.

In [5]:
paths = {
 'canonical_edge': KG_ROOT/'edges/tf_binds_enhancer.parquet',
 'canonical_evidence': KG_ROOT/'evidence/tf_binds_enhancer.parquet',
 'full_support_sidecar': KG_ROOT/'features/remap_crm_tf_enhancer_support_full',
}
for k,p in paths.items():
    print(k, p, 'exists=', p.exists(), 'rows=', parquet_rows(p) if p.exists() else None)

for report in [
 'docs/remap_crm_tf_binds_enhancer_full_feasibility_t_f8cc9e4b.md',
 'docs/remap_crm_tf_binds_enhancer_next_decision_t_2e1b271a.md',
]:
    p=Path(report)
    print('\n---', report, p.exists())
    if p.exists():
        print('\n'.join(p.read_text().splitlines()[:20]))


canonical_edge /Users/jkobject/mnt/gcs/jouvencekb-kg/v2/edges/tf_binds_enhancer.parquet exists= False rows= None
canonical_evidence /Users/jkobject/mnt/gcs/jouvencekb-kg/v2/evidence/tf_binds_enhancer.parquet exists= False rows= None


full_support_sidecar /Users/jkobject/mnt/gcs/jouvencekb-kg/v2/features/remap_crm_tf_enhancer_support_full exists= True rows= 48769967

--- docs/remap_crm_tf_binds_enhancer_full_feasibility_t_f8cc9e4b.md True
# ReMap CRM/peak `tf_binds_enhancer` full-scale feasibility gate

Kanban task: `t_f8cc9e4b`  
Status: `blocked` / `review-required`; staged-only; no canonical writes.

## What I checked

- Accepted bounded pilot `t_a405fe3b`: 1,224,536 edges / 6,356,561 evidence rows from first80 chr1 CRM intervals and bounded all-peak overlap sample.
- Accepted full CRM sidecar `t_5968ce32`: 3,327,980 full CRM intervals, 48,768,788 compact sharded summary rows, and 24,453,482,386 TF × CRM × enhancer candidate support rows explicitly **not materialized**.
- Older bounded all-chromosome support lineage `t_b599d3bb` and current ReMap TODO/status docs.

## Feasibility result

I did **not** write new active `edges/tf_binds_enhancer.parquet` or `evidence/tf_binds_enhancer.parquet` for this card, because

## LaminDB quick probe

In [6]:
import lamindb as ln
print('instance:', ln.setup.settings.instance.slug)
print('modules:', ln.setup.settings.instance.modules)
try:
    import lnschema_txgnn as txs
    for name in ['KGEdge','KGEdgeEvidence']:
        cls = getattr(txs, name)
        try:
            print(name, cls.objects.count())
        except Exception as e:
            print(name, type(e).__name__, str(e)[:200])
except Exception as e:
    print('lnschema_txgnn error:', type(e).__name__, str(e)[:300])

→ connected lamindb: jkobject/jouvencekb


instance: jkobject/jouvencekb
modules: {'pertdb', 'bionty', 'lnschema_txgnn'}
KGEdge 151291
KGEdgeEvidence 109167


## PyG / HeteroData artifacts

In [7]:
for p in sorted(Path('artifacts/staged').glob('**/heterodata/full_graph.metadata.json')):
    try:
        data=json.loads(p.read_text())
    except Exception:
        data={}
    print()
    print(p)
    for key in ['node_counts','edge_counts','relations','build_name','policy']:
        if key in data:
            print(key, data[key])



artifacts/staged/t_015bd9a4_pyg_full_gnn/rep_drug_gene_disease_pathway_pheno/heterodata/full_graph.metadata.json

artifacts/staged/t_857938b0_pyg_embedding_runtime_validation/heterodata/full_graph.metadata.json

artifacts/staged/t_adac88e9_pyg_embedding_runtime_smoke/heterodata/full_graph.metadata.json

artifacts/staged/t_c07b8b57_pyg_dataset_paper_exclusion_smoke/heterodata/full_graph.metadata.json

artifacts/staged/t_c07b8b57_pyg_dataset_paper_include_audit_smoke/heterodata/full_graph.metadata.json

artifacts/staged/t_d97c4547/default_export_dataset_paper_exclusion_smoke/heterodata/full_graph.metadata.json

artifacts/staged/t_fdb7423e_pyg_gnn_embedding_subset/heterodata/full_graph.metadata.json


## Open relation gaps to keep in mind

- `tf_regulates_gene`: schema-only/missing; needs real TF→gene regulatory source.
- `cell_type_expresses_protein`: direct protein measurement only; no RNA projection.
- `transcript_interacts_protein` / `transcript_interacts_gene`: endpoint/source policy unresolved; RBP/RNA sources currently not clean enough.
- `gene_coexpressed_gene`, `disease_comorbid_disease`: currently feature/context unless a sparse source/policy is approved.
- Clinical trials: new layer should include structured fields **and LLM text embeddings** for trial free text / endpoints / failure reasons.

In [8]:
# Show active clinical-trials card if Hermes CLI is available.
import subprocess, shutil
if shutil.which('hermes'):
    out = subprocess.run(['hermes','kanban','--board','txgnn','list','--status','running'], text=True, capture_output=True)
    print(out.stdout[:4000] or out.stderr[:1000])
else:
    print('hermes CLI not found')

Board: txgnn (4 other boards — `hermes kanban boards list`)

● t_524eb18d  running   reviewer              REVIEW t_f65a077a: ClinicalTrials.gov evidence layer prototype + text-embedding gate
● t_b434ee1a  running   tester                EMB-VALIDATE: validate staged foundation embedding manifests/artifacts

